# GeoFormerDock — hoan tat run bi Kaggle timeout tu checkpoint co san

Run Track A goc (geoformerdock, seed 2026) da bi Kaggle "Save & Run All" giet
giua chung sau ~9-12 tieng (het gio phien), nhung da luu duoc mot checkpoint
tot (`best_model.pt`, epoch 44, composite score 0.7980) — chi thieu
`summary.json` (buoc chon nguong + danh gia cuoi cung chua chay).

Notebook nay KHONG train lai tu dau — chi:
1. Copy `data/` (5.5GB) va `best_model.pt` da co san tu Kaggle Dataset ban da
   upload (khong tai lai tu internet — vua nhanh vua dam bao dung y het tap
   receptor da dung luc train that, tranh lech du lieu giua 2 lan chay).
2. Chay `tools/finalize_from_checkpoint.py`: chon pose_threshold tren
   validation, danh gia lai model tren test/val, ghi `summary.json`.

Chi mat vai chuc phut (khong phai vai tieng) vi khong train, chi eval + copy
noi bo tren Kaggle.

**TRUOC KHI CHAY**:
1. Settings (panel phai) → Accelerator → GPU T4 x2, Internet → On.
2. Add Data → gan (attach) dung Kaggle Dataset ban da upload tu file zip
   `_output_` (chua `data/` va `results/models/geoformerdock_valsplit_s2026/`).
   Notebook nay TU DONG do tim duong dan trong `/kaggle/input/`, khong can biet
   ten chinh xac cua dataset.

## 0. Cai dat thu vien truoc tien (khong can restart kernel — xem ly do trong notebook kaggle_verify_and_valsplit.ipynb)

In [ ]:
!pip install -q 'numpy<2' molgrid pytorch-ignite mlflow


In [ ]:
import subprocess, sys
# Kaggle preload numpy rieng vao kernel truoc khi cell nao cua ban chay — kiem
# tra qua subprocess (tien trinh moi, doc numpy tu dia) thay vi import thang.
r = subprocess.run([sys.executable, '-c', '''
import numpy, torch, molgrid
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("molgrid: import OK")
assert numpy.__version__.startswith("1."), f"numpy={numpy.__version__} van >=2"
print("OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr, file=sys.stderr)
    raise RuntimeError('Kiem tra thu vien THAT BAI - dan cho Claude.')


## 1. Lay code moi nhat tu GitHub (can commit moi nhat co `tools/finalize_from_checkpoint.py`)

In [ ]:
import os
if os.path.isdir('/kaggle/working/VNICT2026_Docking_Paper'):
    !cd /kaggle/working/VNICT2026_Docking_Paper && git pull
else:
    !cd /kaggle/working && git clone https://github.com/ducnm-mimhus/VNICT2026_Docking_Paper.git


In [ ]:
%cd /kaggle/working/VNICT2026_Docking_Paper
!git log --oneline -1
!test -f tools/finalize_from_checkpoint.py && echo 'OK: tools/finalize_from_checkpoint.py co san' || echo 'LOI: KHONG thay tools/finalize_from_checkpoint.py - kiem tra lai commit tren GitHub'


## 2. Tim + copy `data/` va `best_model.pt` tu Kaggle Dataset da attach

Tu dong do trong `/kaggle/input/` (khong can biet ten dataset chinh xac) —
tim thu muc `data/` (co `data/types/`) va file
`geoformerdock_valsplit_s2026/best_model.pt` o bat ky do sau nao.

In [ ]:
import glob, os, subprocess

print('=== /kaggle/input ===')
print(subprocess.run(['ls', '-la', '/kaggle/input'], capture_output=True, text=True).stdout)

data_candidates = [
    p for p in glob.glob('/kaggle/input/**/data', recursive=True)
    if os.path.isdir(p) and os.path.isdir(os.path.join(p, 'types'))
]
ckpt_candidates = glob.glob(
    '/kaggle/input/**/geoformerdock_valsplit_s2026/best_model.pt', recursive=True
)

print('data/ candidates:', data_candidates)
print('best_model.pt candidates:', ckpt_candidates)

assert len(data_candidates) >= 1, (
    'Khong tim thay thu muc data/ (co types/) trong /kaggle/input - '
    'kiem tra lai da Add Data dung dataset chua, va dataset co giai nen dung khong.'
)
assert len(ckpt_candidates) >= 1, (
    'Khong tim thay geoformerdock_valsplit_s2026/best_model.pt trong /kaggle/input.'
)

SRC_DATA = data_candidates[0]
SRC_CKPT = ckpt_candidates[0]
print(f'\nDung data/ tu: {SRC_DATA}')
print(f'Dung checkpoint tu: {SRC_CKPT}')


In [ ]:
import shutil, os

DST_DATA = '/kaggle/working/VNICT2026_Docking_Paper/data'
DST_CKPT_DIR = '/kaggle/working/VNICT2026_Docking_Paper/results/models/geoformerdock_valsplit_s2026'

if os.path.isdir(DST_DATA):
    print(f'{DST_DATA} da ton tai - bo qua copy data/ (xoa thu muc do bang tay neu muon copy lai tu dau).')
else:
    print(f'Dang copy {SRC_DATA} -> {DST_DATA} (copy noi bo tren Kaggle, khong qua internet, ~5.5GB, vai phut)...')
    shutil.copytree(SRC_DATA, DST_DATA)
    print('Xong copy data/.')

os.makedirs(DST_CKPT_DIR, exist_ok=True)
shutil.copy2(SRC_CKPT, os.path.join(DST_CKPT_DIR, 'best_model.pt'))
print(f'Da copy checkpoint vao {DST_CKPT_DIR}/best_model.pt')

!ls -la {DST_CKPT_DIR}
!du -sh {DST_DATA}


## 3. Chay finalize (chon nguong + danh gia + ghi summary.json)

Khong train lai — chi 1 luot fit target_normalizer tren train (de dung dung
thong ke chuan hoa nhu luc train that) + 1 luot eval tren val + 1 luot eval
tren test + quet nguong tren val. Xem `tools/finalize_from_checkpoint.py`
de biet chi tiet cac buoc va ly do can khop tham so voi lan train goc.

In [ ]:
!python3 tools/finalize_from_checkpoint.py \
    --checkpoint results/models/geoformerdock_valsplit_s2026/best_model.pt \
    --out_dir results/models/geoformerdock_valsplit_s2026


## 4. Ket qua — dan phan nay vao chat cho Claude

In [ ]:
import json
d = json.load(open('results/models/geoformerdock_valsplit_s2026/summary.json'))
print(json.dumps(d, indent=2, ensure_ascii=False))
